In [1]:
# ============================================================
# UCI-HAR Raw Inertial Data
# Moving-Energy Gated CNN/Mamba vs Full-Mamba Baseline
#
# Proposed:
#   x -> moving-energy gate
#      -> Light CNN for low-energy windows
#      -> Mamba for high-energy windows
#
# Baseline:
#   x -> Full Mamba for all windows
#
# Outputs:
#   1) Proposed gated model performance
#   2) Full-Mamba baseline performance
#   3) Light-route FLOPs
#   4) Heavy-route FLOPs
#   5) Dynamic average FLOPs
#   6) Full-Mamba baseline FLOPs
# ============================================================

from __future__ import annotations

import os
import glob
import zipfile
import urllib.request
import random
import copy
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ============================================================
# 0. Configuration
# ============================================================

SEED = 42

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR"

BATCH_SIZE = 128
EPOCHS_GATED = 80
EPOCHS_FULL_MAMBA = 80

LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

HIDDEN_DIM = 64
MAMBA_BLOCKS = 3
DROPOUT = 0.20

# Top 45% moving-energy windows are routed to Mamba.
# Bottom 55% are routed to Light CNN.
HEAVY_ROUTE_RATIO_TARGET = 0.45

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


# ============================================================
# 2. Download UCI-HAR
# ============================================================

DATA_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://d396qusza40orc.cloudfront.net/getdata%2Fprojectfiles%2FUCI%20HAR%20Dataset.zip",
]


def find_dataset_root(base_dir: str):
    for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == "UCI HAR Dataset":
            inertial_dir = os.path.join(root, "train", "Inertial Signals")
            if os.path.isdir(inertial_dir):
                return root
    return None


def download_and_extract_uci_har(data_dir: str):
    os.makedirs(data_dir, exist_ok=True)

    existing_root = find_dataset_root(data_dir)
    if existing_root is not None:
        print("Dataset already exists:", existing_root)
        return existing_root

    last_error = None

    for i, url in enumerate(DATA_URLS):
        try:
            zip_path = os.path.join(data_dir, f"uci_har_{i}.zip")

            print("Downloading:", url)
            urllib.request.urlretrieve(url, zip_path)

            print("Extracting:", zip_path)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(data_dir)

            nested_zips = glob.glob(os.path.join(data_dir, "**", "*.zip"), recursive=True)
            for nested_zip in nested_zips:
                try:
                    with zipfile.ZipFile(nested_zip, "r") as zf:
                        zf.extractall(data_dir)
                except Exception:
                    pass

            root = find_dataset_root(data_dir)
            if root is not None:
                print("Dataset root:", root)
                return root

        except Exception as e:
            last_error = e
            print("Failed with this URL. Trying next one.")

    raise RuntimeError(f"Could not download or extract UCI-HAR. Last error: {last_error}")


DATA_ROOT = download_and_extract_uci_har(DATA_DIR)


# ============================================================
# 3. Load raw inertial signals
# ============================================================

SIGNAL_FILES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]


def load_txt_matrix(path: str):
    return pd.read_csv(path, sep=r"\s+", header=None).values.astype(np.float32)


def load_split(root: str, split: str):
    signal_dir = os.path.join(root, split, "Inertial Signals")

    signals = []
    for name in SIGNAL_FILES:
        file_path = os.path.join(signal_dir, f"{name}_{split}.txt")
        arr = load_txt_matrix(file_path)
        signals.append(arr)

    X = np.stack(signals, axis=1)  # (N, C, T)

    y_path = os.path.join(root, split, f"y_{split}.txt")
    y = load_txt_matrix(y_path).astype(np.int64).reshape(-1) - 1

    subject_path = os.path.join(root, split, f"subject_{split}.txt")
    subject = load_txt_matrix(subject_path).astype(np.int64).reshape(-1)

    return X, y, subject


def load_activity_names(root: str):
    path = os.path.join(root, "activity_labels.txt")
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["id", "name"])
    return df["name"].tolist()


X_train, y_train, subject_train = load_split(DATA_ROOT, "train")
X_test, y_test, subject_test = load_split(DATA_ROOT, "test")
ACTIVITY_NAMES = load_activity_names(DATA_ROOT)

NUM_CLASSES = len(ACTIVITY_NAMES)
NUM_CHANNELS = X_train.shape[1]
SEQ_LEN = X_train.shape[2]

print("Train X:", X_train.shape)
print("Test X :", X_test.shape)
print("Activities:", ACTIVITY_NAMES)


# ============================================================
# 4. Train-only normalization
# ============================================================

train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Normalized train mean:", float(X_train.mean()))
print("Normalized train std :", float(X_train.std()))


# ============================================================
# 5. Moving-energy gate threshold from training set only
# ============================================================

def compute_moving_energy_np(X: np.ndarray):
    """
    X: (N, C, T)
    Moving energy is calculated from temporal difference.
    """
    dx = X[:, :, 1:] - X[:, :, :-1]
    energy = np.mean(dx ** 2, axis=(1, 2))
    return energy


train_energy = compute_moving_energy_np(X_train)

# Top HEAVY_ROUTE_RATIO_TARGET samples go to Mamba.
gate_tau = float(np.quantile(train_energy, 1.0 - HEAVY_ROUTE_RATIO_TARGET))

train_heavy_ratio = float((train_energy >= gate_tau).mean())

print("Moving-energy gate tau:", gate_tau)
print("Train heavy-route ratio:", train_heavy_ratio)


# ============================================================
# 6. Dataset and dataloader
# ============================================================

class HARWindowDataset(Dataset):
    def __init__(self, X, y, subject=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        if subject is None:
            self.subject = torch.zeros(len(y), dtype=torch.long)
        else:
            self.subject = torch.tensor(subject, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.subject[idx]


train_dataset = HARWindowDataset(X_train, y_train, subject_train)
test_dataset = HARWindowDataset(X_test, y_test, subject_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)


# ============================================================
# 7. Model components
# ============================================================

class MovingEnergyGate(nn.Module):
    """
    Non-parametric gate.
    It does not classify activity.
    It routes each window based on moving energy.
    """
    def __init__(self, tau: float):
        super().__init__()
        self.register_buffer("tau", torch.tensor(float(tau), dtype=torch.float32))

    def forward(self, x):
        # x: (B, C, T)
        dx = x[:, :, 1:] - x[:, :, :-1]
        energy = dx.pow(2).mean(dim=(1, 2))
        route = (energy >= self.tau).long()
        return route, energy


class SeparableConv1DBlock(nn.Module):
    def __init__(self, hidden_dim: int, kernel_size: int = 7, dropout: float = 0.2):
        super().__init__()

        padding = kernel_size // 2

        self.depthwise = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=kernel_size,
            padding=padding,
            groups=hidden_dim,
            bias=False,
        )

        self.pointwise = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=1,
            bias=False,
        )

        self.bn = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = F.silu(x)
        x = self.dropout(x)
        return x + residual


class LightCNNEncoder(nn.Module):
    """
    Light route encoder.
    Used for low moving-energy windows.
    """
    def __init__(self, in_channels: int, hidden_dim: int = 64, dropout: float = 0.2):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.block1 = SeparableConv1DBlock(hidden_dim, kernel_size=7, dropout=dropout)
        self.block2 = SeparableConv1DBlock(hidden_dim, kernel_size=7, dropout=dropout)

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (B, C, T)
        z = self.stem(x)
        z = self.block1(z)
        z = self.block2(z)
        h = z.mean(dim=-1)
        h = self.norm(h)
        return h


class MiniMambaBlock(nn.Module):
    """
    Pure PyTorch Mamba-style selective state-space block.

    This is used for Colab-stable implementation.
    It can be replaced by official mamba_ssm.Mamba if needed.
    """
    def __init__(self, hidden_dim: int = 64, dropout: float = 0.2, conv_kernel: int = 5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.conv_kernel = conv_kernel

        self.norm = nn.LayerNorm(hidden_dim)

        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 2)

        self.depthwise_conv = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=conv_kernel,
            padding=conv_kernel // 2,
            groups=hidden_dim,
            bias=True,
        )

        self.dt_proj = nn.Linear(hidden_dim, hidden_dim)
        self.b_proj = nn.Linear(hidden_dim, hidden_dim)
        self.c_proj = nn.Linear(hidden_dim, hidden_dim)

        self.A_log = nn.Parameter(torch.zeros(hidden_dim))
        self.D = nn.Parameter(torch.ones(hidden_dim))

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def selective_scan(self, u):
        # u: (B, T, H)
        B, T, H = u.shape

        dt = F.softplus(self.dt_proj(u)) + 1e-4
        b_t = torch.tanh(self.b_proj(u))
        c_t = torch.tanh(self.c_proj(u))

        A = torch.exp(self.A_log).view(1, H)

        state = torch.zeros(B, H, device=u.device, dtype=u.dtype)
        outputs = []

        for t in range(T):
            dt_cur = dt[:, t, :]
            u_cur = u[:, t, :]
            b_cur = b_t[:, t, :]
            c_cur = c_t[:, t, :]

            a_bar = torch.exp(-dt_cur * A)
            b_bar = (1.0 - a_bar) * b_cur

            state = a_bar * state + b_bar * u_cur
            y_cur = c_cur * state + self.D.view(1, H) * u_cur

            outputs.append(y_cur)

        y = torch.stack(outputs, dim=1)
        return y

    def forward(self, x):
        # x: (B, T, H)
        residual = x

        x = self.norm(x)

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)

        u = self.depthwise_conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)

        y = self.selective_scan(u)
        y = y * F.silu(z)

        y = self.out_proj(y)
        y = self.dropout(y)

        return residual + y


class MambaEncoder(nn.Module):
    """
    Heavy route encoder.
    Also used as the no-gating full-Mamba baseline.
    """
    def __init__(
        self,
        in_channels: int,
        hidden_dim: int = 64,
        num_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.blocks = nn.ModuleList([
            MiniMambaBlock(hidden_dim=hidden_dim, dropout=dropout, conv_kernel=5)
            for _ in range(num_blocks)
        ])

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        # x: (B, C, T)
        z = self.input_proj(x)       # (B, H, T)
        z = z.transpose(1, 2)        # (B, T, H)

        for block in self.blocks:
            z = block(z)

        z = self.norm(z)
        h = z.mean(dim=1)
        return h


class GatedCNNMambaHAR(nn.Module):
    """
    Proposed model:
        moving-energy gate
        low-energy  -> Light CNN
        high-energy -> Mamba
    """
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        gate_tau: float,
        hidden_dim: int = 64,
        mamba_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.gate = MovingEnergyGate(gate_tau)

        self.light_encoder = LightCNNEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.heavy_encoder = MambaEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            num_blocks=mamba_blocks,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, return_route=False):
        # x: (B, C, T)
        B = x.size(0)

        route, energy = self.gate(x)

        heavy_mask = route.bool()
        light_mask = ~heavy_mask

        H = self.classifier[-1].in_features

        h = torch.zeros(B, H, device=x.device, dtype=x.dtype)

        if light_mask.any():
            h[light_mask] = self.light_encoder(x[light_mask])

        if heavy_mask.any():
            h[heavy_mask] = self.heavy_encoder(x[heavy_mask])

        logits = self.classifier(h)

        if return_route:
            return logits, route, energy

        return logits


class FullMambaHAR(nn.Module):
    """
    Baseline:
        no gate
        all windows go to Mamba
    """
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        mamba_blocks: int = 3,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.encoder = MambaEncoder(
            in_channels=in_channels,
            hidden_dim=hidden_dim,
            num_blocks=mamba_blocks,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        h = self.encoder(x)
        logits = self.classifier(h)
        return logits


# ============================================================
# 8. FLOPs estimation
# ============================================================

def flops_linear(in_dim: int, out_dim: int):
    # Multiply-add counted as 2 FLOPs.
    return 2 * in_dim * out_dim


def flops_conv1d(in_ch: int, out_ch: int, kernel_size: int, out_len: int, groups: int = 1):
    # Multiply-add counted as 2 FLOPs.
    return 2 * out_len * out_ch * (in_ch // groups) * kernel_size


def estimate_gate_flops(C: int, T: int):
    # dx subtraction + square + sum/mean comparison
    # approximate only
    return (T - 1) * C * 3 + 10


def estimate_light_cnn_flops(C: int, T: int, H: int, K: int):
    gate = estimate_gate_flops(C, T)

    stem = flops_conv1d(C, H, kernel_size=3, out_len=T, groups=1)
    stem_bn_act = 5 * T * H

    def sep_block_flops():
        depthwise = flops_conv1d(H, H, kernel_size=7, out_len=T, groups=H)
        pointwise = flops_conv1d(H, H, kernel_size=1, out_len=T, groups=1)
        bn_act_res = 7 * T * H
        return depthwise + pointwise + bn_act_res

    block1 = sep_block_flops()
    block2 = sep_block_flops()

    gap = T * H
    layernorm = 5 * H
    classifier = flops_linear(H, K)

    return gate + stem + stem_bn_act + block1 + block2 + gap + layernorm + classifier


def estimate_mamba_block_flops(T: int, H: int, conv_kernel: int = 5):
    layernorm = 5 * T * H

    in_proj = T * flops_linear(H, 2 * H)
    depthwise_conv = flops_conv1d(H, H, conv_kernel, T, groups=H)

    dt_proj = T * flops_linear(H, H)
    b_proj = T * flops_linear(H, H)
    c_proj = T * flops_linear(H, H)

    selective_scan_elementwise = 35 * T * H

    out_proj = T * flops_linear(H, H)
    residual_add = T * H

    return (
        layernorm
        + in_proj
        + depthwise_conv
        + dt_proj
        + b_proj
        + c_proj
        + selective_scan_elementwise
        + out_proj
        + residual_add
    )


def estimate_mamba_encoder_flops(C: int, T: int, H: int, K: int, blocks: int):
    input_proj = flops_conv1d(C, H, kernel_size=1, out_len=T, groups=1)
    input_bn_act = 5 * T * H

    one_block = estimate_mamba_block_flops(T, H, conv_kernel=5)
    all_blocks = blocks * one_block

    norm = 5 * T * H
    gap = T * H
    classifier = flops_linear(H, K)

    return input_proj + input_bn_act + all_blocks + norm + gap + classifier


def estimate_heavy_route_flops(C: int, T: int, H: int, K: int, blocks: int):
    # Proposed heavy route includes gate overhead + Mamba route.
    return estimate_gate_flops(C, T) + estimate_mamba_encoder_flops(C, T, H, K, blocks)


def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


LIGHT_ROUTE_FLOPS = estimate_light_cnn_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
)

HEAVY_ROUTE_FLOPS = estimate_heavy_route_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    blocks=MAMBA_BLOCKS,
)

FULL_MAMBA_BASELINE_FLOPS = estimate_mamba_encoder_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    blocks=MAMBA_BLOCKS,
)

print("\nApproximate FLOPs per sample")
print("Light route FLOPs        :", f"{LIGHT_ROUTE_FLOPS / 1e6:.4f} M")
print("Heavy Mamba route FLOPs  :", f"{HEAVY_ROUTE_FLOPS / 1e6:.4f} M")
print("Full-Mamba baseline FLOPs:", f"{FULL_MAMBA_BASELINE_FLOPS / 1e6:.4f} M")


# ============================================================
# 9. Training and evaluation utilities
# ============================================================

criterion = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer):
    model.train()

    total_loss = 0.0
    all_y = []
    all_pred = []
    all_route = []

    for batch in loader:
        x, y, _ = batch
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if isinstance(model, GatedCNNMambaHAR):
            logits, route, energy = model(x, return_route=True)
            all_route.append(route.detach().cpu().numpy())
        else:
            logits = model(x)

        loss = criterion(logits, y)
        loss.backward()

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        pred = logits.argmax(dim=1)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_y, all_pred)
    mf1 = f1_score(all_y, all_pred, average="macro")

    route_ratio = None
    if len(all_route) > 0:
        all_route = np.concatenate(all_route)
        route_ratio = float(all_route.mean())

    return avg_loss, acc, mf1, route_ratio


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    all_y = []
    all_pred = []
    all_route = []
    all_energy = []
    all_subject = []

    for batch in loader:
        x, y, subject = batch
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if isinstance(model, GatedCNNMambaHAR):
            logits, route, energy = model(x, return_route=True)
            all_route.append(route.detach().cpu().numpy())
            all_energy.append(energy.detach().cpu().numpy())
        else:
            logits = model(x)

        loss = criterion(logits, y)
        pred = logits.argmax(dim=1)

        total_loss += loss.item() * x.size(0)
        all_y.append(y.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())
        all_subject.append(subject.numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)
    all_subject = np.concatenate(all_subject)

    result = {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro"),
        "y_true": all_y,
        "y_pred": all_pred,
        "subject": all_subject,
    }

    if len(all_route) > 0:
        all_route = np.concatenate(all_route)
        all_energy = np.concatenate(all_energy)

        heavy_ratio = float(all_route.mean())
        light_ratio = 1.0 - heavy_ratio

        avg_flops = (
            light_ratio * LIGHT_ROUTE_FLOPS
            + heavy_ratio * HEAVY_ROUTE_FLOPS
        )

        result.update({
            "route": all_route,
            "energy": all_energy,
            "light_ratio": light_ratio,
            "heavy_ratio": heavy_ratio,
            "avg_flops": avg_flops,
        })
    else:
        result.update({
            "avg_flops": FULL_MAMBA_BASELINE_FLOPS,
        })

    return result


def train_model(model, train_loader, test_loader, epochs, model_name):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
    )

    best_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc, train_mf1, train_route_ratio = train_one_epoch(
            model,
            train_loader,
            optimizer,
        )

        test_result = evaluate_model(model, test_loader)

        scheduler.step()

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_mf1,
            "test_acc": test_result["acc"],
            "test_macro_f1": test_result["macro_f1"],
        }

        if isinstance(model, GatedCNNMambaHAR):
            row["train_heavy_ratio"] = train_route_ratio
            row["test_heavy_ratio"] = test_result["heavy_ratio"]
            row["test_avg_flops_M"] = test_result["avg_flops"] / 1e6
        else:
            row["test_avg_flops_M"] = FULL_MAMBA_BASELINE_FLOPS / 1e6

        history.append(row)

        if test_result["macro_f1"] > best_f1:
            best_f1 = test_result["macro_f1"]
            best_state = copy.deepcopy(model.state_dict())

        if epoch == 1 or epoch % 5 == 0 or epoch == epochs:
            if isinstance(model, GatedCNNMambaHAR):
                print(
                    f"[{model_name}] Epoch {epoch:03d} | "
                    f"Train F1 {train_mf1:.4f} | "
                    f"Test F1 {test_result['macro_f1']:.4f} | "
                    f"Heavy ratio {test_result['heavy_ratio']:.3f} | "
                    f"Avg FLOPs {test_result['avg_flops'] / 1e6:.4f} M"
                )
            else:
                print(
                    f"[{model_name}] Epoch {epoch:03d} | "
                    f"Train F1 {train_mf1:.4f} | "
                    f"Test F1 {test_result['macro_f1']:.4f} | "
                    f"FLOPs {FULL_MAMBA_BASELINE_FLOPS / 1e6:.4f} M"
                )

    model.load_state_dict(best_state)
    final_result = evaluate_model(model, test_loader)

    return model, pd.DataFrame(history), final_result


# ============================================================
# 10. Train proposed gated CNN/Mamba model
# ============================================================

set_seed(SEED)

gated_model = GatedCNNMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    gate_tau=gate_tau,
    hidden_dim=HIDDEN_DIM,
    mamba_blocks=MAMBA_BLOCKS,
    dropout=DROPOUT,
).to(DEVICE)

print("\n================ Proposed Gated CNN/Mamba Model ================")
print(gated_model)
print("Trainable parameters:", count_params(gated_model))

gated_model, gated_history, gated_result = train_model(
    model=gated_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS_GATED,
    model_name="Gated-CNN-Mamba",
)


# ============================================================
# 11. Train full-Mamba baseline
# ============================================================

set_seed(SEED)

full_mamba_model = FullMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    mamba_blocks=MAMBA_BLOCKS,
    dropout=DROPOUT,
).to(DEVICE)

print("\n================ Full-Mamba Baseline ================")
print(full_mamba_model)
print("Trainable parameters:", count_params(full_mamba_model))

full_mamba_model, full_history, full_result = train_model(
    model=full_mamba_model,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS_FULL_MAMBA,
    model_name="Full-Mamba",
)


# ============================================================
# 12. Final comparison
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "Proposed Gated CNN/Mamba",
        "Accuracy": gated_result["acc"],
        "Macro-F1": gated_result["macro_f1"],
        "Light-route ratio": gated_result["light_ratio"],
        "Heavy-route ratio": gated_result["heavy_ratio"],
        "Avg FLOPs (M)": gated_result["avg_flops"] / 1e6,
        "Light route FLOPs (M)": LIGHT_ROUTE_FLOPS / 1e6,
        "Heavy route FLOPs (M)": HEAVY_ROUTE_FLOPS / 1e6,
        "Full-Mamba baseline FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Params": count_params(gated_model),
    },
    {
        "Model": "Full-Mamba Baseline",
        "Accuracy": full_result["acc"],
        "Macro-F1": full_result["macro_f1"],
        "Light-route ratio": 0.0,
        "Heavy-route ratio": 1.0,
        "Avg FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Light route FLOPs (M)": None,
        "Heavy route FLOPs (M)": None,
        "Full-Mamba baseline FLOPs (M)": FULL_MAMBA_BASELINE_FLOPS / 1e6,
        "Params": count_params(full_mamba_model),
    },
])

print("\n================ Final Comparison ================")
try:
    display(comparison)
except Exception:
    print(comparison.to_string(index=False))


# ============================================================
# 13. Route analysis for proposed model
# ============================================================

route_df = pd.DataFrame({
    "true_label": gated_result["y_true"],
    "pred_label": gated_result["y_pred"],
    "route": gated_result["route"],
    "moving_energy": gated_result["energy"],
})

route_df["activity"] = route_df["true_label"].apply(lambda i: ACTIVITY_NAMES[i])
route_df["route_name"] = route_df["route"].apply(lambda r: "Heavy-Mamba" if r == 1 else "Light-CNN")
route_df["correct"] = route_df["true_label"] == route_df["pred_label"]

route_summary = (
    route_df
    .groupby("activity")
    .agg(
        samples=("activity", "count"),
        light_route_ratio=("route", lambda x: float((x == 0).mean())),
        heavy_route_ratio=("route", "mean"),
        avg_moving_energy=("moving_energy", "mean"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
    .sort_values("heavy_route_ratio", ascending=False)
)

print("\n================ Proposed Model Route Summary by Activity ================")
try:
    display(route_summary)
except Exception:
    print(route_summary.to_string(index=False))


# ============================================================
# 14. Classification reports
# ============================================================

print("\n================ Classification Report: Proposed Gated CNN/Mamba ================")
print(
    classification_report(
        gated_result["y_true"],
        gated_result["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)

print("\n================ Classification Report: Full-Mamba Baseline ================")
print(
    classification_report(
        full_result["y_true"],
        full_result["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)


# ============================================================
# 15. Confusion matrices
# ============================================================

gated_cm = confusion_matrix(gated_result["y_true"], gated_result["y_pred"])
full_cm = confusion_matrix(full_result["y_true"], full_result["y_pred"])

gated_cm_df = pd.DataFrame(
    gated_cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

full_cm_df = pd.DataFrame(
    full_cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

print("\n================ Confusion Matrix: Proposed Gated CNN/Mamba ================")
try:
    display(gated_cm_df)
except Exception:
    print(gated_cm_df.to_string())

print("\n================ Confusion Matrix: Full-Mamba Baseline ================")
try:
    display(full_cm_df)
except Exception:
    print(full_cm_df.to_string())


# ============================================================
# 16. Save models and results
# ============================================================

SAVE_DIR = "/content/har_gated_cnn_mamba_results"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(
    {
        "model_state_dict": gated_model.state_dict(),
        "gate_tau": gate_tau,
        "train_mean": train_mean,
        "train_std": train_std,
        "config": {
            "model": "GatedCNNMambaHAR",
            "hidden_dim": HIDDEN_DIM,
            "mamba_blocks": MAMBA_BLOCKS,
            "dropout": DROPOUT,
            "heavy_route_ratio_target": HEAVY_ROUTE_RATIO_TARGET,
            "activity_names": ACTIVITY_NAMES,
        },
    },
    os.path.join(SAVE_DIR, "gated_cnn_mamba.pt"),
)

torch.save(
    {
        "model_state_dict": full_mamba_model.state_dict(),
        "train_mean": train_mean,
        "train_std": train_std,
        "config": {
            "model": "FullMambaHAR",
            "hidden_dim": HIDDEN_DIM,
            "mamba_blocks": MAMBA_BLOCKS,
            "dropout": DROPOUT,
            "activity_names": ACTIVITY_NAMES,
        },
    },
    os.path.join(SAVE_DIR, "full_mamba_baseline.pt"),
)

comparison.to_csv(os.path.join(SAVE_DIR, "comparison.csv"), index=False)
route_summary.to_csv(os.path.join(SAVE_DIR, "route_summary.csv"), index=False)
gated_history.to_csv(os.path.join(SAVE_DIR, "gated_history.csv"), index=False)
full_history.to_csv(os.path.join(SAVE_DIR, "full_mamba_history.csv"), index=False)

print("\nSaved results to:", SAVE_DIR)


# ============================================================
# 17. Single-window inference for proposed model
# ============================================================

@torch.no_grad()
def predict_one_window_gated(model, x_np):
    """
    x_np: shape (C, T), already normalized.
    """
    model.eval()

    x = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    logits, route, energy = model(x, return_route=True)

    prob = F.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    pred_id = int(prob.argmax())

    route_id = int(route.item())
    route_name = "Heavy-Mamba" if route_id == 1 else "Light-CNN"

    used_flops = HEAVY_ROUTE_FLOPS if route_id == 1 else LIGHT_ROUTE_FLOPS

    return {
        "pred_id": pred_id,
        "pred_activity": ACTIVITY_NAMES[pred_id],
        "confidence": float(prob[pred_id]),
        "route": route_name,
        "moving_energy": float(energy.item()),
        "used_flops_M": used_flops / 1e6,
    }


sample_idx = 0

example = predict_one_window_gated(gated_model, X_test[sample_idx])

print("\n================ Single-window Example: Proposed Model ================")
print("True activity:", ACTIVITY_NAMES[y_test[sample_idx]])
print("Prediction   :", example["pred_activity"])
print("Route        :", example["route"])
print("Energy       :", example["moving_energy"])
print("Used FLOPs   :", f"{example['used_flops_M']:.4f} M")

Device: cuda
Dataset already exists: /content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset
Train X: (7352, 9, 128)
Test X : (2947, 9, 128)
Activities: ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']
Normalized train mean: 0.00015565917419735342
Normalized train std : 1.000395655632019
Moving-energy gate tau: 0.02595616690814495
Train heavy-route ratio: 0.4499455930359086

Approximate FLOPs per sample
Light route FLOPs        : 2.9373 M
Heavy Mamba route FLOPs  : 20.3695 M
Full-Mamba baseline FLOPs: 20.3661 M

================ Proposed Gated CNN/Mamba Model ================
GatedCNNMambaHAR(
  (gate): MovingEnergyGate()
  (light_encoder): LightCNNEncoder(
    (stem): Sequential(
      (0): Conv1d(9, 64, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU()
    )
    (block1): SeparableConv1DBlock(
      (depthwise): Conv1

,Model,Accuracy,Macro-F1,Light-route ratio,Heavy-route ratio,Avg FLOPs (M),Light route FLOPs (M),Heavy route FLOPs (M),Full-Mamba baseline FLOPs (M),Params
0,Proposed Gated CNN/Mamba,0.959620,0.959067,0.523923,0.476077,11.236366,2.937263,20.369519,20.36608,89478
1,Full-Mamba Baseline,0.958602,0.958834,0.000000,1.000000,20.366080,NaN,NaN,20.36608,78150



================ Proposed Model Route Summary by Activity ================


,activity,samples,light_route_ratio,heavy_route_ratio,avg_moving_energy,accuracy
4,WALKING_DOWNSTAIRS,420,0.000000,1.000000,0.395032,0.995238
3,WALKING,496,0.000000,1.000000,0.323742,0.951613
5,WALKING_UPSTAIRS,471,0.000000,1.000000,0.182885,0.946921
2,STANDING,532,0.977444,0.022556,0.002304,0.964286
0,LAYING,537,0.994413,0.005587,0.001330,1.000000
1,SITTING,491,0.997963,0.002037,0.000959,0.900204



================ Classification Report: Proposed Gated CNN/Mamba ================
                    precision    recall  f1-score   support

           WALKING     1.0000    0.9516    0.9752       496
  WALKING_UPSTAIRS     0.9911    0.9469    0.9685       471
WALKING_DOWNSTAIRS     0.8951    0.9952    0.9425       420
           SITTING     0.9609    0.9002    0.9295       491
          STANDING     0.9161    0.9643    0.9396       532
            LAYING     0.9981    1.0000    0.9991       537

          accuracy                         0.9596      2947
         macro avg     0.9602    0.9597    0.9591      2947
      weighted avg     0.9616    0.9596    0.9598      2947


================ Classification Report: Full-Mamba Baseline ================
                    precision    recall  f1-score   support

           WALKING     0.9610    0.9940    0.9772       496
  WALKING_UPSTAIRS     0.9780    0.9427    0.9600       471
WALKING_DOWNSTAIRS     0.9810    0.9833    0.9822      

,pred_WALKING,pred_WALKING_UPSTAIRS,pred_WALKING_DOWNSTAIRS,pred_SITTING,pred_STANDING,pred_LAYING
true_WALKING,472,0,24,0,0,0
true_WALKING_UPSTAIRS,0,446,25,0,0,0
true_WALKING_DOWNSTAIRS,0,2,418,0,0,0
true_SITTING,0,1,0,442,47,1
true_STANDING,0,1,0,18,513,0
true_LAYING,0,0,0,0,0,537



================ Confusion Matrix: Full-Mamba Baseline ================


,pred_WALKING,pred_WALKING_UPSTAIRS,pred_WALKING_DOWNSTAIRS,pred_SITTING,pred_STANDING,pred_LAYING
true_WALKING,493,0,2,0,1,0
true_WALKING_UPSTAIRS,20,444,6,0,1,0
true_WALKING_DOWNSTAIRS,0,7,413,0,0,0
true_SITTING,0,3,0,429,59,0
true_STANDING,0,0,0,23,509,0
true_LAYING,0,0,0,0,0,537



Saved results to: /content/har_gated_cnn_mamba_results

================ Single-window Example: Proposed Model ================
True activity: STANDING
Prediction   : STANDING
Route        : Light-CNN
Energy       : 0.00621773861348629
Used FLOPs   : 2.9373 M


In [2]:
# ============================================================
# Confidence-Gated Shared-Mamba HAR on UCI-HAR Raw Inertial Data
# Google Colab-ready full code
#
# Idea:
#   Same Mamba backbone, different executed depth.
#
#   x -> Input Projection -> Mamba Block 1 -> Early Classifier
#      -> if confidence high: stop
#      -> if confidence low : execute remaining Mamba blocks
#
# Routes:
#   Early route: fewer Mamba blocks, lower FLOPs
#   Full route : more Mamba blocks, higher FLOPs
#
# ============================================================

from __future__ import annotations

import os
import glob
import zipfile
import urllib.request
import random
import copy
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


# ============================================================
# 0. Configuration
# ============================================================

SEED = 42

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/confidence_gated_shared_mamba_uci_har.pt"

BATCH_SIZE = 128
EPOCHS = 80
LR = 2e-3
WEIGHT_DECAY = 1e-2
GRAD_CLIP = 1.0

HIDDEN_DIM = 64
TOTAL_MAMBA_BLOCKS = 3
EARLY_EXIT_BLOCKS = 1

DROPOUT = 0.20
EARLY_LOSS_WEIGHT = 0.50

# Validation-based confidence threshold calibration.
# This limits the maximum fraction of samples routed to the full path.
MAX_FULL_ROUTE_RATIO = 0.45

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)


# ============================================================
# 1. Reproducibility
# ============================================================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


set_seed(SEED)


# ============================================================
# 2. Download and extract UCI-HAR dataset
# ============================================================

DATA_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://d396qusza40orc.cloudfront.net/getdata%2Fprojectfiles%2FUCI%20HAR%20Dataset.zip",
]


def find_dataset_root(base_dir: str):
    for root, dirs, files in os.walk(base_dir):
        if os.path.basename(root) == "UCI HAR Dataset":
            inertial_dir = os.path.join(root, "train", "Inertial Signals")
            if os.path.isdir(inertial_dir):
                return root
    return None


def download_and_extract_uci_har(data_dir: str):
    os.makedirs(data_dir, exist_ok=True)

    existing_root = find_dataset_root(data_dir)
    if existing_root is not None:
        print("Dataset already exists:", existing_root)
        return existing_root

    last_error = None

    for i, url in enumerate(DATA_URLS):
        try:
            zip_path = os.path.join(data_dir, f"uci_har_{i}.zip")

            print("Downloading:", url)
            urllib.request.urlretrieve(url, zip_path)

            print("Extracting:", zip_path)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(data_dir)

            nested_zips = glob.glob(os.path.join(data_dir, "**", "*.zip"), recursive=True)
            for nested_zip in nested_zips:
                try:
                    with zipfile.ZipFile(nested_zip, "r") as zf:
                        zf.extractall(data_dir)
                except Exception:
                    pass

            root = find_dataset_root(data_dir)
            if root is not None:
                print("Dataset root:", root)
                return root

        except Exception as e:
            last_error = e
            print("Failed with this URL. Trying next one.")

    raise RuntimeError(f"Could not download or extract UCI-HAR dataset. Last error: {last_error}")


DATA_ROOT = download_and_extract_uci_har(DATA_DIR)


# ============================================================
# 3. Load UCI-HAR raw inertial signals
# ============================================================

SIGNAL_FILES = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
]


def load_txt_matrix(path: str):
    return pd.read_csv(path, sep=r"\s+", header=None).values.astype(np.float32)


def load_split(root: str, split: str):
    signal_dir = os.path.join(root, split, "Inertial Signals")

    signals = []
    for name in SIGNAL_FILES:
        file_path = os.path.join(signal_dir, f"{name}_{split}.txt")
        arr = load_txt_matrix(file_path)
        signals.append(arr)

    X = np.stack(signals, axis=1)  # (N, C, T)

    y_path = os.path.join(root, split, f"y_{split}.txt")
    y = load_txt_matrix(y_path).astype(np.int64).reshape(-1) - 1

    subject_path = os.path.join(root, split, f"subject_{split}.txt")
    subject = load_txt_matrix(subject_path).astype(np.int64).reshape(-1)

    return X, y, subject


def load_activity_names(root: str):
    path = os.path.join(root, "activity_labels.txt")
    df = pd.read_csv(path, sep=r"\s+", header=None, names=["id", "name"])
    return df["name"].tolist()


X_train_full, y_train_full, subject_train_full = load_split(DATA_ROOT, "train")
X_test, y_test, subject_test = load_split(DATA_ROOT, "test")
ACTIVITY_NAMES = load_activity_names(DATA_ROOT)

NUM_CLASSES = len(ACTIVITY_NAMES)
NUM_CHANNELS = X_train_full.shape[1]
SEQ_LEN = X_train_full.shape[2]

print("Full train X:", X_train_full.shape)
print("Test X      :", X_test.shape)
print("Activities  :", ACTIVITY_NAMES)


# ============================================================
# 4. Subject-disjoint split inside official training set
#    Train subjects: model fitting
#    Val subjects  : confidence threshold calibration
# ============================================================

def subject_disjoint_train_val_split(X, y, subject, val_ratio=0.20, seed=42):
    unique_subjects = np.array(sorted(np.unique(subject)))
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_subjects)

    n_val = max(1, int(round(len(unique_subjects) * val_ratio)))
    val_subjects = set(unique_subjects[:n_val].tolist())

    val_mask = np.array([s in val_subjects for s in subject])
    train_mask = ~val_mask

    return (
        X[train_mask], y[train_mask], subject[train_mask],
        X[val_mask], y[val_mask], subject[val_mask],
        sorted(list(val_subjects)),
    )


X_train, y_train, subject_train, X_val, y_val, subject_val, val_subjects = subject_disjoint_train_val_split(
    X_train_full,
    y_train_full,
    subject_train_full,
    val_ratio=0.20,
    seed=SEED,
)

print("Train subjects for fitting:", sorted(np.unique(subject_train).tolist()))
print("Val subjects for gate calibration:", val_subjects)
print("Train X:", X_train.shape)
print("Val X  :", X_val.shape)


# ============================================================
# 5. Train-only normalization
# ============================================================

train_mean = X_train.mean(axis=(0, 2), keepdims=True)
train_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

print("Normalized train mean:", float(X_train.mean()))
print("Normalized train std :", float(X_train.std()))


# ============================================================
# 6. Dataset and dataloader
# ============================================================

class HARWindowDataset(Dataset):
    def __init__(self, X, y, subject=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

        if subject is None:
            self.subject = torch.zeros(len(y), dtype=torch.long)
        else:
            self.subject = torch.tensor(subject, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.subject[idx]


train_dataset = HARWindowDataset(X_train, y_train, subject_train)
val_dataset = HARWindowDataset(X_val, y_val, subject_val)
test_dataset = HARWindowDataset(X_test, y_test, subject_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=True,
)


# ============================================================
# 7. Mamba-style block
# ============================================================

class MiniMambaBlock(nn.Module):
    """
    Pure PyTorch Mamba-style selective state-space block.

    This implementation is designed for Colab reproducibility.
    It follows the key Mamba-style idea:
      - input projection
      - depthwise temporal convolution
      - input-dependent selective state update
      - gated output projection

    For an official paper implementation, this block can be replaced with
    mamba_ssm.Mamba while keeping the same routing structure.
    """
    def __init__(self, hidden_dim: int, dropout: float = 0.2, conv_kernel: int = 5):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.conv_kernel = conv_kernel

        self.norm = nn.LayerNorm(hidden_dim)

        self.in_proj = nn.Linear(hidden_dim, hidden_dim * 2)

        self.depthwise_conv = nn.Conv1d(
            hidden_dim,
            hidden_dim,
            kernel_size=conv_kernel,
            padding=conv_kernel // 2,
            groups=hidden_dim,
            bias=True,
        )

        self.dt_proj = nn.Linear(hidden_dim, hidden_dim)
        self.b_proj = nn.Linear(hidden_dim, hidden_dim)
        self.c_proj = nn.Linear(hidden_dim, hidden_dim)

        self.A_log = nn.Parameter(torch.zeros(hidden_dim))
        self.D = nn.Parameter(torch.ones(hidden_dim))

        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def selective_scan(self, u):
        # u: (B, T, H)
        B, T, H = u.shape

        dt = F.softplus(self.dt_proj(u)) + 1e-4
        b_t = torch.tanh(self.b_proj(u))
        c_t = torch.tanh(self.c_proj(u))

        A = torch.exp(self.A_log).view(1, H)

        state = torch.zeros(B, H, device=u.device, dtype=u.dtype)
        outputs = []

        for t in range(T):
            dt_cur = dt[:, t, :]
            u_cur = u[:, t, :]
            b_cur = b_t[:, t, :]
            c_cur = c_t[:, t, :]

            a_bar = torch.exp(-dt_cur * A)
            b_bar = (1.0 - a_bar) * b_cur

            state = a_bar * state + b_bar * u_cur
            y_cur = c_cur * state + self.D.view(1, H) * u_cur

            outputs.append(y_cur)

        y = torch.stack(outputs, dim=1)
        return y

    def forward(self, x):
        # x: (B, T, H)
        residual = x

        x = self.norm(x)

        xz = self.in_proj(x)
        u, z = xz.chunk(2, dim=-1)

        u = self.depthwise_conv(u.transpose(1, 2)).transpose(1, 2)
        u = F.silu(u)

        y = self.selective_scan(u)
        y = y * F.silu(z)

        y = self.out_proj(y)
        y = self.dropout(y)

        return residual + y


# ============================================================
# 8. Confidence-gated shared Mamba model
# ============================================================

class ConfidenceGatedSharedMambaHAR(nn.Module):
    """
    Same Mamba backbone, two execution depths.

    Early route:
        input projection + first Mamba block + shared classifier

    Full route:
        input projection + all Mamba blocks + shared classifier

    The confidence gate is based on early classifier confidence.
    """
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        hidden_dim: int = 64,
        total_blocks: int = 3,
        early_exit_blocks: int = 1,
        dropout: float = 0.2,
    ):
        super().__init__()

        assert 1 <= early_exit_blocks < total_blocks

        self.in_channels = in_channels
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        self.total_blocks = total_blocks
        self.early_exit_blocks = early_exit_blocks

        self.input_proj = nn.Sequential(
            nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
        )

        self.blocks = nn.ModuleList([
            MiniMambaBlock(hidden_dim=hidden_dim, dropout=dropout, conv_kernel=5)
            for _ in range(total_blocks)
        ])

        self.readout_norm = nn.LayerNorm(hidden_dim)

        # Shared classifier for both early and full routes.
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def pool_and_classify(self, z):
        # z: (B, T, H)
        z = self.readout_norm(z)
        h = z.mean(dim=1)
        logits = self.classifier(h)
        return logits

    def forward_all(self, x):
        """
        Training forward.
        Always computes both early logits and final logits.
        """
        # x: (B, C, T)
        z = self.input_proj(x)
        z = z.transpose(1, 2)  # (B, T, H)

        early_logits = None

        for i, block in enumerate(self.blocks):
            z = block(z)

            if i + 1 == self.early_exit_blocks:
                early_logits = self.pool_and_classify(z)

        final_logits = self.pool_and_classify(z)

        return early_logits, final_logits

    @torch.no_grad()
    def forward_dynamic(self, x, confidence_tau: float):
        """
        Inference forward with confidence-based routing.
        """
        z = self.input_proj(x)
        z = z.transpose(1, 2)

        for i in range(self.early_exit_blocks):
            z = self.blocks[i](z)

        early_logits = self.pool_and_classify(z)
        early_prob = F.softmax(early_logits, dim=1)
        confidence, early_pred = early_prob.max(dim=1)

        full_mask = confidence < confidence_tau

        output_logits = early_logits.clone()
        route = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        if full_mask.any():
            z_full = z[full_mask]

            for i in range(self.early_exit_blocks, self.total_blocks):
                z_full = self.blocks[i](z_full)

            full_logits = self.pool_and_classify(z_full)

            output_logits[full_mask] = full_logits
            route[full_mask] = 1

        return output_logits, route, confidence


model = ConfidenceGatedSharedMambaHAR(
    in_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    hidden_dim=HIDDEN_DIM,
    total_blocks=TOTAL_MAMBA_BLOCKS,
    early_exit_blocks=EARLY_EXIT_BLOCKS,
    dropout=DROPOUT,
).to(DEVICE)

print(model)


# ============================================================
# 9. Parameter count
# ============================================================

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


print("Total trainable parameters:", count_params(model))
print("Input projection parameters:", count_params(model.input_proj))
print("Mamba block parameters:", count_params(model.blocks))
print("Shared classifier parameters:", count_params(model.classifier))


# ============================================================
# 10. FLOPs estimation
# ============================================================

def flops_linear(in_dim, out_dim):
    # multiply-add counted as 2 FLOPs
    return 2 * in_dim * out_dim


def flops_conv1d(in_ch, out_ch, kernel_size, out_len, groups=1):
    # multiply-add counted as 2 FLOPs
    return 2 * out_len * out_ch * (in_ch // groups) * kernel_size


def estimate_minimamba_block_flops(T, H, conv_kernel=5):
    """
    Approximate per-sample FLOPs for MiniMambaBlock.

    Includes:
      LayerNorm approximation
      Linear H -> 2H
      Depthwise Conv1D
      dt, B, C projections
      selective scan elementwise operations
      output projection
      residual addition
    """
    layernorm = 5 * T * H

    in_proj = T * flops_linear(H, 2 * H)
    depthwise_conv = flops_conv1d(H, H, conv_kernel, T, groups=H)

    dt_proj = T * flops_linear(H, H)
    b_proj = T * flops_linear(H, H)
    c_proj = T * flops_linear(H, H)

    selective_scan_elementwise = 35 * T * H

    out_proj = T * flops_linear(H, H)
    residual_add = T * H

    total = (
        layernorm
        + in_proj
        + depthwise_conv
        + dt_proj
        + b_proj
        + c_proj
        + selective_scan_elementwise
        + out_proj
        + residual_add
    )

    return total


def estimate_route_flops(
    C,
    T,
    H,
    K,
    total_blocks,
    early_blocks,
    conv_kernel=5,
):
    input_proj = flops_conv1d(C, H, kernel_size=1, out_len=T, groups=1)

    block_flops = estimate_minimamba_block_flops(T=T, H=H, conv_kernel=conv_kernel)

    readout_norm = 5 * T * H
    gap = T * H
    classifier = flops_linear(H, K)

    early_head = readout_norm + gap + classifier
    full_head = readout_norm + gap + classifier

    early_route = (
        input_proj
        + early_blocks * block_flops
        + early_head
    )

    full_route = (
        input_proj
        + early_blocks * block_flops
        + early_head
        + (total_blocks - early_blocks) * block_flops
        + full_head
    )

    full_model_once = (
        input_proj
        + total_blocks * block_flops
        + full_head
    )

    return {
        "input_projection": input_proj,
        "one_mamba_block": block_flops,
        "early_route": early_route,
        "full_route_dynamic": full_route,
        "full_model_once": full_model_once,
    }


flops_info = estimate_route_flops(
    C=NUM_CHANNELS,
    T=SEQ_LEN,
    H=HIDDEN_DIM,
    K=NUM_CLASSES,
    total_blocks=TOTAL_MAMBA_BLOCKS,
    early_blocks=EARLY_EXIT_BLOCKS,
    conv_kernel=5,
)

print("\nApproximate per-sample FLOPs")
print("Input projection FLOPs :", f"{flops_info['input_projection'] / 1e6:.4f} M")
print("One Mamba block FLOPs  :", f"{flops_info['one_mamba_block'] / 1e6:.4f} M")
print("Early route FLOPs      :", f"{flops_info['early_route'] / 1e6:.4f} M")
print("Full route FLOPs       :", f"{flops_info['full_route_dynamic'] / 1e6:.4f} M")
print("Full model once FLOPs  :", f"{flops_info['full_model_once'] / 1e6:.4f} M")


# ============================================================
# 11. Training utilities
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
)


def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    all_y = []
    all_early_pred = []
    all_final_pred = []

    for x, y, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        early_logits, final_logits = model.forward_all(x)

        early_loss = criterion(early_logits, y)
        final_loss = criterion(final_logits, y)

        loss = final_loss + EARLY_LOSS_WEIGHT * early_loss

        loss.backward()

        if GRAD_CLIP is not None:
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()

        total_loss += loss.item() * x.size(0)

        early_pred = early_logits.argmax(dim=1)
        final_pred = final_logits.argmax(dim=1)

        all_y.append(y.detach().cpu().numpy())
        all_early_pred.append(early_pred.detach().cpu().numpy())
        all_final_pred.append(final_pred.detach().cpu().numpy())

    all_y = np.concatenate(all_y)
    all_early_pred = np.concatenate(all_early_pred)
    all_final_pred = np.concatenate(all_final_pred)

    avg_loss = total_loss / len(loader.dataset)
    early_acc = accuracy_score(all_y, all_early_pred)
    early_mf1 = f1_score(all_y, all_early_pred, average="macro")
    final_acc = accuracy_score(all_y, all_final_pred)
    final_mf1 = f1_score(all_y, all_final_pred, average="macro")

    return avg_loss, early_acc, early_mf1, final_acc, final_mf1


@torch.no_grad()
def evaluate_all(model, loader):
    model.eval()

    total_loss = 0.0
    all_y = []
    all_early_pred = []
    all_final_pred = []
    all_conf = []
    all_subject = []

    for x, y, subject in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        early_logits, final_logits = model.forward_all(x)

        early_loss = criterion(early_logits, y)
        final_loss = criterion(final_logits, y)
        loss = final_loss + EARLY_LOSS_WEIGHT * early_loss

        total_loss += loss.item() * x.size(0)

        early_prob = F.softmax(early_logits, dim=1)
        confidence, early_pred = early_prob.max(dim=1)

        final_pred = final_logits.argmax(dim=1)

        all_y.append(y.detach().cpu().numpy())
        all_early_pred.append(early_pred.detach().cpu().numpy())
        all_final_pred.append(final_pred.detach().cpu().numpy())
        all_conf.append(confidence.detach().cpu().numpy())
        all_subject.append(subject.numpy())

    all_y = np.concatenate(all_y)
    all_early_pred = np.concatenate(all_early_pred)
    all_final_pred = np.concatenate(all_final_pred)
    all_conf = np.concatenate(all_conf)
    all_subject = np.concatenate(all_subject)

    return {
        "loss": total_loss / len(loader.dataset),
        "y_true": all_y,
        "early_pred": all_early_pred,
        "final_pred": all_final_pred,
        "confidence": all_conf,
        "subject": all_subject,
        "early_acc": accuracy_score(all_y, all_early_pred),
        "early_macro_f1": f1_score(all_y, all_early_pred, average="macro"),
        "final_acc": accuracy_score(all_y, all_final_pred),
        "final_macro_f1": f1_score(all_y, all_final_pred, average="macro"),
    }


def dynamic_metrics_from_outputs(y_true, early_pred, final_pred, confidence, tau):
    full_mask = confidence < tau

    dynamic_pred = early_pred.copy()
    dynamic_pred[full_mask] = final_pred[full_mask]

    acc = accuracy_score(y_true, dynamic_pred)
    mf1 = f1_score(y_true, dynamic_pred, average="macro")
    full_ratio = float(full_mask.mean())

    avg_flops = (
        (1.0 - full_ratio) * flops_info["early_route"]
        + full_ratio * flops_info["full_route_dynamic"]
    )

    return {
        "tau": float(tau),
        "acc": acc,
        "macro_f1": mf1,
        "full_route_ratio": full_ratio,
        "avg_flops": avg_flops,
        "dynamic_pred": dynamic_pred,
        "full_mask": full_mask,
    }


def calibrate_confidence_threshold(
    y_true,
    early_pred,
    final_pred,
    confidence,
    max_full_route_ratio=0.45,
):
    candidate_taus = np.unique(
        np.quantile(confidence, np.linspace(0.01, 0.99, 199))
    )

    records = []

    for tau in candidate_taus:
        m = dynamic_metrics_from_outputs(
            y_true=y_true,
            early_pred=early_pred,
            final_pred=final_pred,
            confidence=confidence,
            tau=tau,
        )
        records.append(m)

    feasible = [
        r for r in records
        if r["full_route_ratio"] <= max_full_route_ratio
    ]

    if len(feasible) > 0:
        best = sorted(
            feasible,
            key=lambda r: (r["macro_f1"], -r["avg_flops"]),
            reverse=True,
        )[0]
    else:
        best = sorted(
            records,
            key=lambda r: (r["macro_f1"], -r["avg_flops"]),
            reverse=True,
        )[0]

    return best, records


@torch.no_grad()
def evaluate_dynamic(model, loader, confidence_tau):
    model.eval()

    all_y = []
    all_pred = []
    all_route = []
    all_conf = []
    all_subject = []

    for x, y, subject in loader:
        x = x.to(DEVICE, non_blocking=True)

        logits, route, confidence = model.forward_dynamic(
            x,
            confidence_tau=confidence_tau,
        )

        pred = logits.argmax(dim=1)

        all_y.append(y.numpy())
        all_pred.append(pred.detach().cpu().numpy())
        all_route.append(route.detach().cpu().numpy())
        all_conf.append(confidence.detach().cpu().numpy())
        all_subject.append(subject.numpy())

    all_y = np.concatenate(all_y)
    all_pred = np.concatenate(all_pred)
    all_route = np.concatenate(all_route)
    all_conf = np.concatenate(all_conf)
    all_subject = np.concatenate(all_subject)

    full_ratio = float(all_route.mean())

    avg_flops = (
        (1.0 - full_ratio) * flops_info["early_route"]
        + full_ratio * flops_info["full_route_dynamic"]
    )

    return {
        "y_true": all_y,
        "y_pred": all_pred,
        "route": all_route,
        "confidence": all_conf,
        "subject": all_subject,
        "acc": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro"),
        "full_route_ratio": full_ratio,
        "early_route_ratio": 1.0 - full_ratio,
        "avg_flops": avg_flops,
    }


# ============================================================
# 12. Train
# ============================================================

best_val_final_mf1 = -1.0
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_early_acc, train_early_mf1, train_final_acc, train_final_mf1 = train_one_epoch(
        model,
        train_loader,
    )

    val_result = evaluate_all(model, val_loader)

    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_early_acc": train_early_acc,
        "train_early_macro_f1": train_early_mf1,
        "train_final_acc": train_final_acc,
        "train_final_macro_f1": train_final_mf1,
        "val_early_acc": val_result["early_acc"],
        "val_early_macro_f1": val_result["early_macro_f1"],
        "val_final_acc": val_result["final_acc"],
        "val_final_macro_f1": val_result["final_macro_f1"],
        "lr": scheduler.get_last_lr()[0],
    }

    history.append(row)

    if val_result["final_macro_f1"] > best_val_final_mf1:
        best_val_final_mf1 = val_result["final_macro_f1"]
        best_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(
            f"Epoch {epoch:03d} | "
            f"Train early F1 {train_early_mf1:.4f} final F1 {train_final_mf1:.4f} | "
            f"Val early F1 {val_result['early_macro_f1']:.4f} "
            f"final F1 {val_result['final_macro_f1']:.4f}"
        )

model.load_state_dict(best_state)

print("\nBest validation final Macro-F1:", best_val_final_mf1)


# ============================================================
# 13. Calibrate confidence threshold on validation set
# ============================================================

val_outputs = evaluate_all(model, val_loader)

best_tau_record, tau_records = calibrate_confidence_threshold(
    y_true=val_outputs["y_true"],
    early_pred=val_outputs["early_pred"],
    final_pred=val_outputs["final_pred"],
    confidence=val_outputs["confidence"],
    max_full_route_ratio=MAX_FULL_ROUTE_RATIO,
)

confidence_tau = best_tau_record["tau"]

print("\nCalibrated confidence threshold")
print("Tau:", confidence_tau)
print("Val dynamic Acc:", best_tau_record["acc"])
print("Val dynamic Macro-F1:", best_tau_record["macro_f1"])
print("Val full-route ratio:", best_tau_record["full_route_ratio"])
print("Val average FLOPs:", f"{best_tau_record['avg_flops'] / 1e6:.4f} M")

tau_df = pd.DataFrame([
    {
        "tau": r["tau"],
        "macro_f1": r["macro_f1"],
        "acc": r["acc"],
        "full_route_ratio": r["full_route_ratio"],
        "avg_flops_M": r["avg_flops"] / 1e6,
    }
    for r in tau_records
])

tau_df = tau_df.sort_values(["macro_f1", "avg_flops_M"], ascending=[False, True])

print("\nTop threshold candidates:")
try:
    display(tau_df.head(10))
except Exception:
    print(tau_df.head(10).to_string(index=False))


# ============================================================
# 14. Final test evaluation
# ============================================================

test_all = evaluate_all(model, test_loader)
test_dynamic = evaluate_dynamic(model, test_loader, confidence_tau=confidence_tau)

print("\n================ Final Test Results ================")
print("Early-only Acc      :", test_all["early_acc"])
print("Early-only Macro-F1 :", test_all["early_macro_f1"])
print("Full-only Acc       :", test_all["final_acc"])
print("Full-only Macro-F1  :", test_all["final_macro_f1"])
print("Dynamic Acc         :", test_dynamic["acc"])
print("Dynamic Macro-F1    :", test_dynamic["macro_f1"])

print("\n================ Route FLOPs ================")
print("Early route FLOPs per sample :", f"{flops_info['early_route'] / 1e6:.4f} M")
print("Full route FLOPs per sample  :", f"{flops_info['full_route_dynamic'] / 1e6:.4f} M")
print("Full model once FLOPs        :", f"{flops_info['full_model_once'] / 1e6:.4f} M")
print("Dynamic avg FLOPs per sample :", f"{test_dynamic['avg_flops'] / 1e6:.4f} M")

print("\n================ Route Ratio ================")
print("Early route ratio:", test_dynamic["early_route_ratio"])
print("Full route ratio :", test_dynamic["full_route_ratio"])

print("\nClassification Report: Dynamic Routing")
print(
    classification_report(
        test_dynamic["y_true"],
        test_dynamic["y_pred"],
        target_names=ACTIVITY_NAMES,
        digits=4,
    )
)

cm = confusion_matrix(test_dynamic["y_true"], test_dynamic["y_pred"])
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{name}" for name in ACTIVITY_NAMES],
    columns=[f"pred_{name}" for name in ACTIVITY_NAMES],
)

print("\nConfusion Matrix: Dynamic Routing")
try:
    display(cm_df)
except Exception:
    print(cm_df.to_string())


# ============================================================
# 15. Route analysis by activity class
# ============================================================

route_df = pd.DataFrame({
    "true_label": test_dynamic["y_true"],
    "pred_label": test_dynamic["y_pred"],
    "route": test_dynamic["route"],
    "confidence": test_dynamic["confidence"],
})

route_df["activity"] = route_df["true_label"].apply(lambda i: ACTIVITY_NAMES[i])
route_df["route_name"] = route_df["route"].apply(lambda r: "Full-Mamba" if r == 1 else "Early-Mamba")
route_df["correct"] = route_df["true_label"] == route_df["pred_label"]

route_summary = (
    route_df
    .groupby("activity")
    .agg(
        samples=("activity", "count"),
        full_route_ratio=("route", "mean"),
        avg_early_confidence=("confidence", "mean"),
        dynamic_accuracy=("correct", "mean"),
    )
    .reset_index()
    .sort_values("full_route_ratio", ascending=False)
)

print("\nRoute Summary by Activity")
try:
    display(route_summary)
except Exception:
    print(route_summary.to_string(index=False))


# ============================================================
# 16. Save model
# ============================================================

save_obj = {
    "model_state_dict": model.state_dict(),
    "config": {
        "model": "ConfidenceGatedSharedMambaHAR",
        "num_channels": NUM_CHANNELS,
        "seq_len": SEQ_LEN,
        "num_classes": NUM_CLASSES,
        "activity_names": ACTIVITY_NAMES,
        "hidden_dim": HIDDEN_DIM,
        "total_mamba_blocks": TOTAL_MAMBA_BLOCKS,
        "early_exit_blocks": EARLY_EXIT_BLOCKS,
        "dropout": DROPOUT,
        "early_loss_weight": EARLY_LOSS_WEIGHT,
        "confidence_tau": confidence_tau,
        "max_full_route_ratio": MAX_FULL_ROUTE_RATIO,
        "flops_info": flops_info,
    },
    "train_mean": train_mean,
    "train_std": train_std,
}

torch.save(save_obj, MODEL_SAVE_PATH)
print("\nSaved model to:", MODEL_SAVE_PATH)


# ============================================================
# 17. Single-window inference example
# ============================================================

@torch.no_grad()
def predict_one_window(model, x_np, confidence_tau):
    """
    x_np: shape (C, T), already normalized.
    """
    model.eval()

    x = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)

    logits, route, confidence = model.forward_dynamic(
        x,
        confidence_tau=confidence_tau,
    )

    prob = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred_id = int(prob.argmax())

    route_id = int(route.item())
    route_name = "Full-Mamba" if route_id == 1 else "Early-Mamba"

    used_flops = (
        flops_info["full_route_dynamic"]
        if route_id == 1
        else flops_info["early_route"]
    )

    return {
        "pred_id": pred_id,
        "pred_activity": ACTIVITY_NAMES[pred_id],
        "confidence_from_early_head": float(confidence.item()),
        "route": route_name,
        "used_flops_M": used_flops / 1e6,
        "probability": prob,
    }


sample_idx = 0
example_result = predict_one_window(
    model,
    X_test[sample_idx],
    confidence_tau=confidence_tau,
)

print("\nSingle-window inference example")
print("True activity:", ACTIVITY_NAMES[y_test[sample_idx]])
print("Prediction:", example_result["pred_activity"])
print("Route:", example_result["route"])
print("Early confidence:", example_result["confidence_from_early_head"])
print("Used FLOPs:", f"{example_result['used_flops_M']:.4f} M")

Device: cuda
Dataset already exists: /content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/UCI HAR Dataset
Full train X: (7352, 9, 128)
Test X      : (2947, 9, 128)
Activities  : ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS', 'SITTING', 'STANDING', 'LAYING']
Train subjects for fitting: [1, 3, 5, 6, 7, 8, 11, 14, 15, 16, 19, 21, 22, 23, 27, 28, 29]
Val subjects for gate calibration: [17, 25, 26, 30]
Train X: (5800, 9, 128)
Val X  : (1552, 9, 128)
Normalized train mean: 2.4871607820387e-05
Normalized train std : 1.0003135204315186
ConfidenceGatedSharedMambaHAR(
  (input_proj): Sequential(
    (0): Conv1d(9, 64, kernel_size=(1,), stride=(1,), bias=False)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU()
  )
  (blocks): ModuleList(
    (0-2): 3 x MiniMambaBlock(
      (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (in_proj): Linear(in_features=64, out_features=128, bias=True)
      (depthwise_conv): Conv1d(6

,tau,macro_f1,acc,full_route_ratio,avg_flops_M
0,0.580243,0.968631,0.966495,0.010309,7.045474
1,0.638671,0.968010,0.965851,0.015464,7.114898
3,0.710882,0.968010,0.965851,0.025129,7.245070
6,0.846302,0.967986,0.965851,0.039948,7.444666
2,0.691424,0.967404,0.965206,0.019974,7.175645
4,0.753730,0.967388,0.965206,0.030284,7.314495
5,0.803744,0.967388,0.965206,0.034794,7.375242
7,0.856192,0.967363,0.965206,0.045103,7.514091
8,0.870766,0.967363,0.965206,0.049613,7.574838
9,0.900260,0.966740,0.964562,0.054768,7.644263



================ Final Test Results ================
Early-only Acc      : 0.9365456396335257
Early-only Macro-F1 : 0.9362798384027475
Full-only Acc       : 0.9429928741092637
Full-only Macro-F1  : 0.9432152205295264
Dynamic Acc         : 0.9389209365456397
Dynamic Macro-F1    : 0.9389121920947435

================ Route FLOPs ================
Early route FLOPs per sample : 6.9066 M
Full route FLOPs per sample  : 20.3750 M
Full model once FLOPs        : 20.3251 M
Dynamic avg FLOPs per sample : 7.1169 M

================ Route Ratio ================
Early route ratio: 0.9843909060061079
Full route ratio : 0.015609093993892093

Classification Report: Dynamic Routing
                    precision    recall  f1-score   support

           WALKING     0.9773    0.9556    0.9664       496
  WALKING_UPSTAIRS     0.9911    0.9406    0.9651       471
WALKING_DOWNSTAIRS     0.9204    0.9905    0.9541       420
           SITTING     0.9034    0.8187    0.8590       491
          STANDING     0.

,pred_WALKING,pred_WALKING_UPSTAIRS,pred_WALKING_DOWNSTAIRS,pred_SITTING,pred_STANDING,pred_LAYING
true_WALKING,474,0,22,0,0,0
true_WALKING_UPSTAIRS,7,443,14,7,0,0
true_WALKING_DOWNSTAIRS,3,1,416,0,0,0
true_SITTING,0,3,0,402,81,5
true_STANDING,1,0,0,36,495,0
true_LAYING,0,0,0,0,0,537



Route Summary by Activity


,activity,samples,full_route_ratio,avg_early_confidence,dynamic_accuracy
5,WALKING_UPSTAIRS,471,0.038217,0.957912,0.940552
1,SITTING,491,0.022403,0.959003,0.818737
2,STANDING,532,0.016917,0.972497,0.930451
3,WALKING,496,0.012097,0.977952,0.955645
4,WALKING_DOWNSTAIRS,420,0.004762,0.982685,0.990476
0,LAYING,537,0.000000,0.999978,1.000000



Saved model to: /content/drive/MyDrive/Colab Notebooks/UCI-HAR/UCI-HAR/confidence_gated_shared_mamba_uci_har.pt

Single-window inference example
True activity: STANDING
Prediction: STANDING
Route: Early-Mamba
Early confidence: 0.9998579025268555
Used FLOPs: 6.9066 M
